# Fix: Cross-Species Directional Concordance Column

## What this notebook fixes

The binomial test in NB04 reads from `Results/common_genes.csv` and looks for a column called `Direction_Concordant`.  
That column was never written to the file — so the test defaulted to `n_concordant = 0`, producing `p = 1.0`.

**Actual data:** All 13 cross-species shared genes are upregulated in both human and canine AD lesions.  
After this fix, the correct result will be `n_concordant = 13/13`, `p ≈ 0.000122`.

## What this notebook does
1. Reads `Results/common_genes.csv`
2. Adds `Direction_Concordant` column (`True` where `sign(Human_log2FC) == sign(Canine_log2FC)`)
3. Saves the updated file back to `Results/common_genes.csv`
4. Recomputes the binomial test and saves the corrected `Results/cross_species_binomial.json`
5. Prints a summary

**Run this notebook BEFORE re-running NB04**, so that NB04 reads the corrected file.

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import pandas as pd
import numpy as np
import json
import os
from scipy.stats import binomtest

# ── Path configuration ─────────────────────────────────────────────────────
# Adjust BASE to match your Google Drive path
BASE = '/content/drive/MyDrive/Research/Atopic Dermatitis/Research Paper'
RES  = os.path.join(BASE, 'Results')

print('Base path:', BASE)
print('Results path:', RES)

Base path: /content/drive/MyDrive/Research/Atopic Dermatitis/Research Paper
Results path: /content/drive/MyDrive/Research/Atopic Dermatitis/Research Paper/Results


In [3]:
# ── Step 1: Load common_genes.csv ──────────────────────────────────────────
common_path = os.path.join(RES, 'common_genes.csv')

if not os.path.exists(common_path):
    raise FileNotFoundError(f'common_genes.csv not found at: {common_path}')

common_df = pd.read_csv(common_path)
print(f'Loaded common_genes.csv: {len(common_df)} rows')
print('Columns:', common_df.columns.tolist())
print()
print(common_df[['Gene_Symbol', 'Human_log2FC', 'Canine_log2FC']].to_string(index=False))

Loaded common_genes.csv: 13 rows
Columns: ['Gene_Symbol', 'Human_Status', 'Canine_Status', 'Human_log2FC', 'Canine_log2FC', 'Both_Novel', 'Both_Known']

Gene_Symbol  Human_log2FC  Canine_log2FC
      CCL17        2.7517          4.200
      CCL19        2.3390          2.795
       CCL2        2.6398          2.545
      CXCL8        3.9605          2.590
       IL13        3.7817          2.450
    IL13RA2        2.5961          3.170
        IL6        3.7045          3.270
       RGS1        2.5310          2.115
    S100A12        3.6846          2.715
     S100A8        3.5234          2.505
     S100A9        4.2841          3.650
     SAMSN1        2.5774          3.545
   SERPINB4        4.9500          2.700


In [4]:
# ── Step 2: Add Direction_Concordant column ────────────────────────────────
# A gene is directionally concordant if the sign of log2FC is the same
# in both human and canine (both upregulated or both downregulated)

common_df['Direction_Concordant'] = (
    np.sign(common_df['Human_log2FC'].astype(float)) ==
    np.sign(common_df['Canine_log2FC'].astype(float))
)

print('Direction concordance per gene:')
print(common_df[['Gene_Symbol', 'Human_log2FC', 'Canine_log2FC', 'Direction_Concordant']].to_string(index=False))
print()
print(f'Concordant: {common_df["Direction_Concordant"].sum()} / {len(common_df)}')

Direction concordance per gene:
Gene_Symbol  Human_log2FC  Canine_log2FC  Direction_Concordant
      CCL17        2.7517          4.200                  True
      CCL19        2.3390          2.795                  True
       CCL2        2.6398          2.545                  True
      CXCL8        3.9605          2.590                  True
       IL13        3.7817          2.450                  True
    IL13RA2        2.5961          3.170                  True
        IL6        3.7045          3.270                  True
       RGS1        2.5310          2.115                  True
    S100A12        3.6846          2.715                  True
     S100A8        3.5234          2.505                  True
     S100A9        4.2841          3.650                  True
     SAMSN1        2.5774          3.545                  True
   SERPINB4        4.9500          2.700                  True

Concordant: 13 / 13


In [5]:
# ── Step 3: Save updated common_genes.csv ─────────────────────────────────
common_df.to_csv(common_path, index=False)
print(f'Saved updated common_genes.csv to: {common_path}')

# Also save a copy to NB03/Tables if it exists there
nb03_tab = os.path.join(RES, 'NB03', 'Tables')
nb03_path = os.path.join(nb03_tab, 'cross_species_common_genes.csv')
if os.path.exists(nb03_tab):
    common_df.to_csv(nb03_path, index=False)
    print(f'Also saved to: {nb03_path}')

Saved updated common_genes.csv to: /content/drive/MyDrive/Research/Atopic Dermatitis/Research Paper/Results/common_genes.csv
Also saved to: /content/drive/MyDrive/Research/Atopic Dermatitis/Research Paper/Results/NB03/Tables/cross_species_common_genes.csv


In [6]:
# ── Step 4: Recompute binomial test ────────────────────────────────────────
# Test: is the fraction of concordant genes significantly greater than 50%?
# H0: each gene has 50% probability of being concordant (random)
# H1: fraction > 50% (directional conservation is real)

# Only count rows where both FCs are non-null and non-zero
valid = common_df.dropna(subset=['Human_log2FC', 'Canine_log2FC'])
n_total    = len(valid)
n_concordant = int(valid['Direction_Concordant'].sum())

binom_result = binomtest(n_concordant, n_total, p=0.5, alternative='greater')

print('=' * 50)
print('CORRECTED CROSS-SPECIES BINOMIAL TEST')
print('=' * 50)
print(f'  Concordant genes  : {n_concordant} / {n_total}')
print(f'  Binomial p-value  : {binom_result.pvalue:.6f}')
print(f'  Null hypothesis   : 50% concordance by chance')
print(f'  Alternative       : greater than 50%')
if binom_result.pvalue < 0.05:
    print(f'  Result            : SIGNIFICANT — cross-species direction is conserved')
else:
    print(f'  Result            : NOT significant')

CORRECTED CROSS-SPECIES BINOMIAL TEST
  Concordant genes  : 13 / 13
  Binomial p-value  : 0.000122
  Null hypothesis   : 50% concordance by chance
  Alternative       : greater than 50%
  Result            : SIGNIFICANT — cross-species direction is conserved


In [7]:
# ── Step 5: Save corrected cross_species_binomial.json ────────────────────
binom_out = {
    'n_concordant' : n_concordant,
    'n_total'      : n_total,
    'p_value'      : round(binom_result.pvalue, 6),
    'note'         : 'Fixed: Direction_Concordant column added to common_genes.csv. '
                     'Previous result (n_concordant=0, p=1.0) was a code artifact '
                     '(column was absent, defaulted to 0).'
}

out_path = os.path.join(RES, 'cross_species_binomial.json')
json.dump(binom_out, open(out_path, 'w'), indent=2)
print(f'Saved corrected binomial result to: {out_path}')
print()
print('File contents:')
print(json.dumps(binom_out, indent=2))

Saved corrected binomial result to: /content/drive/MyDrive/Research/Atopic Dermatitis/Research Paper/Results/cross_species_binomial.json

File contents:
{
  "n_concordant": 13,
  "n_total": 13,
  "p_value": 0.000122,
  "note": "Fixed: Direction_Concordant column added to common_genes.csv. Previous result (n_concordant=0, p=1.0) was a code artifact (column was absent, defaulted to 0)."
}


In [8]:
# ── Summary ────────────────────────────────────────────────────────────────
print('=' * 60)
print('FIX COMPLETE')
print('=' * 60)
print(f'  common_genes.csv  : Direction_Concordant column added')
print(f'  Concordant        : {n_concordant}/{n_total} genes')
print(f'  Binomial p        : {binom_result.pvalue:.6f}')
print()
print('Next step: Re-run NB04_Validation.ipynb')
print('NB04 will now correctly read Direction_Concordant and compute the binomial test.')

FIX COMPLETE
  common_genes.csv  : Direction_Concordant column added
  Concordant        : 13/13 genes
  Binomial p        : 0.000122

Next step: Re-run NB04_Validation.ipynb
NB04 will now correctly read Direction_Concordant and compute the binomial test.
